<a href="https://colab.research.google.com/github/yueteng9511/Neural_Network/blob/main/%E7%A5%9E%E7%B6%93%E7%B6%B2%E8%B7%AF01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 關於 AI 協助說明
本次作業撰寫過程中，針對神經網路架構原理、程式碼各行意義、
以及debug（如程式碼縮排錯誤）等部分，曾使用 Claude 進行提問
與討論，協助釐清觀念。以下表格整理主要討論主題與重點。

| 討論主題 | 我問的內容 | AI 說明重點（濃縮） ||
|---|---|---|---|
| 全連結層 | 全連結是什麼 | 上一層每個神經元都連到下一層每個神經元，連線數=兩層神經元數相乘 | |
| Dense層與參數量 | model.add(Dense(...))這行在幹嘛 | 建立全連結層，神經元數決定輸出維度，input_dim只有第一層要寫 | |
| ReLU | activation='relu'是什麼、有什麼用 | max(0,x)，讓網路具備非線性能力，避免疊多層仍等同線性函數 | |
| softmax | 輸出層為何用softmax | 把原始分數轉成加總為1的機率分布，才能跟one-hot答案比較 | |
| loss/compile | model.compile在做什麼 | 設定訓練規則:loss函數、optimizer、要額外追蹤的指標 | |
| categorical_crossentropy | 這是什麼、為何比mse適合分類問題 | 只關注正確答案位置的機率，用log懲罰沒信心的錯誤，比mse更貼合分類任務目標 | |
| epoch與batch | epoch是什麼、32筆一批是什麼意思 | epoch=看過全部資料一次；batch=一次平行處理的資料筆數，用於平均誤差、更新參數 | |
| 驗證資料 | 驗證資料在幹嘛、跟測試資料差在哪 | 驗證資料訓練過程中監控用、可重複使用但會間接影響決策；測試資料只在最後用一次，保持客觀 | |
| accuracy | accuracy是什麼、為何不能拿來訓練 | 猜對比例，是離散指標無法微分，只能當觀察用，不能驅動梯度下降 | |
| 實驗設計 | 層數/loss function怎麼比較才合理 | 控制變因原則:一次只變一個因素,才能判斷差異是哪個因素造成的 | |

In [1]:
!pip install gradio

In [2]:
%matplotlib inline

# 標準數據分析、畫圖套件
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 神經網路方面
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

# 互動設計用
from ipywidgets import interact_manual

# 神速打造 web app 的 Gradio
import gradio as gr

###先把MNIST資料集下載下來，並一次拆成訓練集和測試集。
###x是圖片(像素28*28)，y是答案(0~9以評分)。
註：MNIST 是一份早就準備好、公開釋出的標準資料集，這份資料集裡：訓練集固定就是 60,000 張手寫數字圖片，測試集固定就是 10,000 張。

In [3]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
print(f'訓練資料總筆數為 {len(x_train)} 筆資料')
print(f'測試資料總筆數為 {len(x_test)} 筆資料')

訓練資料總筆數為 60000 筆資料
測試資料總筆數為 10000 筆資料


###給開發者確認第n張資料長什麼樣子以及官方標記的正確答案。

In [5]:
def show_xy(n=0):
    X = x_train[n]
    plt.xticks([], [])
    plt.yticks([], [])
    plt.imshow(X, cmap = 'Greys')
    print(f'本資料 y 給定的答案為: {y_train[n]}')

###把剛剛定義好的 show_xy 函式,包裝成一個帶滑桿的互動介面,讓你在 Colab 裡可以用拖曳滑桿的方式,即時切換要看第幾張圖片,而不用每次都手動改程式碼重新執行。

In [6]:
interact_manual(show_xy, n=(0,59999));

interactive(children=(IntSlider(value=0, description='n', max=59999), Button(description='Run Interact', style…

###印出訓練集裡第 n 張圖片,實際儲存的原始數字矩陣

In [7]:
def show_data(n = 100):
    X = x_train[n]
    print(X)

###把剛剛定義好的 show_data 函式,包裝成一個帶滑桿的互動介面

In [8]:
interact_manual(show_data, n=(0,59999));

interactive(children=(IntSlider(value=100, description='n', max=59999), Button(description='Run Interact', sty…

###把兩份圖片資料，各自做同樣的兩個處理——攤平成一維向量、把數值壓縮到 0~1 之間。
註：神經網路訓練時，數值的尺度（scale）如果太大，容易讓訓練過程中的梯度計算變得不穩定（數值忽大忽小，參數更新的步伐也會跟著劇烈震盪），把輸入統一縮放到一個較小、一致的範圍（像 0~1），能讓訓練過程更穩定、收斂更快。這是深度學習裡一個很常見的前處理慣例。

In [9]:
x_train = x_train.reshape(60000, 784)/255
x_test = x_test.reshape(10000, 784)/255

###把 y_train、y_test 裡「單一數字」形式的答案，轉換成之前提過的 one-hot 編碼格式
註：one-hot編碼是指把整數 k，轉換成一個長度 10 的向量，只有第 k 個位置是 1，其餘全部是 0（位置編號從 0 開始算)。

In [10]:
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

###Sequential 是一個「裝著神經網路層物件、並且知道怎麼依序串接、訓練、使用這些層」的容器

###model.add(...):
往sequential裡加入第一個元素的動作。
###Dense(20, ...):
建立一個全連結層，32代表這一層有32個神經元。784×32=25088條連線（權重），再加上32個bias，這一層共25120個要學的參數。
###input_dim=784:
這是只有第一層才需要寫的參數，告訴 Keras：「即將流進來的輸入，每一筆是 784 個數字」。之後第二層開始，Keras 自動知道輸入維度就是「上一層的神經元數」，不用再手動講。
###activation='relu':
ReLU：max(0,𝑥)，讓這一層具備非線性能力，避免整個網路退化成單純的線性函數。
###Dense(10, ...):
此為輸出層，因最後輸出為0~9共十種，因此神經元數為10。
###activation='softmax':
把這一層算出來的 10 個原始數字，轉換成「加起來等於1」的機率分布，才能被解讀成「模型認為這張圖是各個數字的機率各是多少」，也才能跟 one-hot 編碼過的 y_train（同樣是10維、加起來等於1的向量）互相比較、計算損失。
###model.compile(...):
它不會加任何新的層，只是給這個已經搭好的架構,配置好訓練時要用的規則。
###loss='mse':
mse（均方誤差）會拿模型輸出的10個機率、跟 y_train 的10維 one-hot 答案逐項相減、平方、取平均
###optimizer=SGD(learning_rate=0.087):
（Stochastic Gradient Descent）是決定「算出誤差之後，具體怎麼調整那784×32個權重、還有其他層的權重跟bias」的演算法。learning_rate=0.087 則是每一次調整的步伐大小
###metrics=['accuracy']:
loss（mse算出來的那個誤差值）雖然是真正拿去驅動訓練的東西，但這個數字本身很抽象、不容易直覺感受「模型到底準不準」；accuracy（正確率）則是人類更容易理解的指標——猜對的比例是多少。
###model.fit():
把整理好的訓練資料（x_train 已經 reshape+正規化、y_train 已經 one-hot）餵給 model（已經疊好 A 組架構、也 compile 過），開始跑訓練迴圈。
###epochs=10:
代表 x_train 這 6 萬筆資料，會被完整看過 10 遍。每一遍內部，會依照預設 batch_size=32，切成約 1875 批，每批算完平均誤差就更新一次參數。
###validation_split=0.1:
這裡會自動從 x_train 裡切出 10%（6000筆）當驗證集，這 6000 筆完全不參與訓練，只在每個 epoch 結束後拿出來打分數（同時算 val_loss 跟 val_accuracy，因為compile時有設定 metrics=['accuracy']）。
###history_A.history:
history_A 是 model.fit(...) 執行完的回傳值。它裡面有一個叫 .history 的屬性（attribute），本質上是一個字典（dictionary），長得像這樣（示意）：
history_A.history = {
    'loss':          [0.0812, 0.0598, ..., 0.0234],   # 10個數字,每個epoch一個
    'accuracy':      [0.3241, 0.5872, ..., 0.9012],
    'val_loss':      [0.0654, 0.0521, ..., 0.0245],
    'val_accuracy':  [0.4523, 0.6234, ..., 0.8956]
}


## 實驗一：隱藏層層數比較（2層 vs 4層 vs 5層）

本次作業要求隱藏層不能是3層，因此設計了三組不同層數的架構進行
比較：A組(2層)、B組(4層)、C組(5層)，每層皆固定為32個神經元、
activation皆使用relu，loss、optimizer等其他設定也全部保持一致，
只讓「層數」這一個變因改變，才能單純觀察層數對結果的影響。

選擇這三組層數的原因：MNIST是相對單純的分類任務，理論上不需要
非常深的網路就能達到不錯的正確率，因此想觀察隨著層數增加（從2層
到5層），驗證正確率是否會持續提升，或是在某個層數之後就不再有
明顯幫助，甚至因為模型過於複雜而不利於訓練。

##實驗一：A組(2層，每層32個神經元)

In [11]:
model = Sequential()
model.add(Dense(32, input_dim=784, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(10, activation='softmax'))
model.compile(loss='mse', optimizer=SGD(learning_rate=0.087), metrics=['accuracy'])
history_A = model.fit(x_train, y_train, epochs=10, validation_split=0.1)
acc_A = max(history_A.history['val_accuracy'])
print(f"A組 最佳驗證正確率: {acc_A*100:.2f}%")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.4474 - loss: 0.0730 - val_accuracy: 0.7575 - val_loss: 0.0441
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8150 - loss: 0.0306 - val_accuracy: 0.8913 - val_loss: 0.0180
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8814 - loss: 0.0189 - val_accuracy: 0.9127 - val_loss: 0.0139
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8993 - loss: 0.0159 - val_accuracy: 0.9215 - val_loss: 0.0123
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9082 - loss: 0.0143 - val_accuracy: 0.9277 - val_loss: 0.0113
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9155 - loss: 0.0132 - val_accuracy: 0.9352 - val_loss: 0.0105
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9201 - loss: 0.0125 - val_accuracy: 0.9377 - val_loss: 0.0099
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9246 - loss: 0.0118 -

##實驗一：(4層，每層32個神經元)`

In [12]:
model = Sequential()
model.add(Dense(32, input_dim=784, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(10, activation='softmax'))
model.compile(loss='mse', optimizer=SGD(learning_rate=0.087), metrics=['accuracy'])
history_B = model.fit(x_train, y_train, epochs=10, validation_split=0.1)
acc_B = max(history_B.history['val_accuracy'])
print(f"B組 最佳驗證正確率: {acc_B*100:.2f}%")

Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.2954 - loss: 0.0848 - val_accuracy: 0.5407 - val_loss: 0.0630
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7455 - loss: 0.0378 - val_accuracy: 0.8618 - val_loss: 0.0215
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8685 - loss: 0.0202 - val_accuracy: 0.9095 - val_loss: 0.0140
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8955 - loss: 0.0160 - val_accuracy: 0.9227 - val_loss: 0.0120
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9092 - loss: 0.0141 - val_accuracy: 0.9258 - val_loss: 0.0110
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9173 - loss: 0.0128 - val_accuracy: 0.9353 - val_loss: 0.0100
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9245 - loss: 0.0118 - val_accuracy: 0.9373 - val_loss: 0.0094
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9301 - loss: 0.0109 - 

##實驗一：(5層，每層32個神經元)

In [13]:
model = Sequential()
model.add(Dense(32, input_dim=784, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(10, activation='softmax'))
model.compile(loss='mse', optimizer=SGD(learning_rate=0.087), metrics=['accuracy'])
history_C = model.fit(x_train, y_train, epochs=10, validation_split=0.1)
acc_C = max(history_C.history['val_accuracy'])
print(f"C組 最佳驗證正確率: {acc_C*100:.2f}%")

Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.2110 - loss: 0.0885 - val_accuracy: 0.3293 - val_loss: 0.0832
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.5367 - loss: 0.0596 - val_accuracy: 0.8405 - val_loss: 0.0291
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8493 - loss: 0.0232 - val_accuracy: 0.8972 - val_loss: 0.0161
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8892 - loss: 0.0170 - val_accuracy: 0.9198 - val_loss: 0.0128
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9053 - loss: 0.0145 - val_accuracy: 0.9265 - val_loss: 0.0113
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9160 - loss: 0.0130 - val_accuracy: 0.9327 - val_loss: 0.0101
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9228 - loss: 0.0118 - val_accuracy: 0.9380 - val_loss: 0.0095
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9297 - loss: 0.0110 - 

In [14]:
print("===== 層數比較總結 =====")
print(f"A組(2層): {acc_A*100:.2f}%")
print(f"B組(4層): {acc_B*100:.2f}%")
print(f"C組(5層): {acc_C*100:.2f}%")

===== 層數比較總結 =====
A組(2層): 94.40%
B組(4層): 94.98%
C組(5層): 95.12%


### 觀察與心得
本次比較2層、4層、5層三種隱藏層層數(每層皆32個神經元、relu)，
驗證正確率分別為94.40%、94.98%、95.12%，其中C組(5層)略高，
但三組差距僅約0.72%，差距不算顯著。

值得一提的是，這三組數字在不同次執行時曾出現過名次互換的情況
（例如另一次執行中B組表現最高），差距同樣都在1%以內。這進一步
印證了：由於權重初始化與資料洗牌具有隨機性，這樣小幅度的差距
很可能主要來自隨機波動，而非層數本身造成的顯著差異。也就是說，
就本次任務規模而言，2~5層之間的差異對模型表現的影響相對有限。

## 實驗二：Loss Function 比較（mse vs categorical_crossentropy）

延續實驗一選出的B組架構（4層，每層32神經元），固定層數、神經元數、
optimizer不變，只改變loss function，觀察對驗證正確率的影響。

選擇比較這兩種loss function的原因：mse（均方誤差）原本是設計給
「迴歸問題」使用的損失函數，用來衡量連續數值之間的距離；而這次
的任務是「分類問題」（判斷圖片屬於0~9哪一類），categorical_
crossentropy是專門為多類別分類問題設計的損失函數，理論上更適合
這種情境。因此想實際驗證：換成理論上更合適的loss function，是否
真的能帶來實際的正確率提升。

In [15]:
model = Sequential()
model.add(Dense(32, input_dim=784, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(10, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer=SGD(learning_rate=0.087), metrics=['accuracy'])
history_E = model.fit(x_train, y_train, epochs=10, validation_split=0.1)
acc_E = max(history_E.history['val_accuracy'])
print(f"E組(4層,categorical_crossentropy) 最佳驗證正確率: {acc_E*100:.2f}%")

Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8621 - loss: 0.4343 - val_accuracy: 0.9488 - val_loss: 0.1745
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9449 - loss: 0.1801 - val_accuracy: 0.9507 - val_loss: 0.1556
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9558 - loss: 0.1428 - val_accuracy: 0.9677 - val_loss: 0.1086
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9631 - loss: 0.1200 - val_accuracy: 0.9708 - val_loss: 0.0994
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9678 - loss: 0.1047 - val_accuracy: 0.9582 - val_loss: 0.1506
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9717 - loss: 0.0921 - val_accuracy: 0.9462 - val_loss: 0.1751
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9735 - loss: 0.0848 - val_accuracy: 0.9598 - val_loss: 0.1414
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9760 - loss: 0.0773 - 

In [16]:
print("===== 層數比較總結 =====")
print(f"A組(2層): {acc_A*100:.2f}%")
print(f"B組(4層): {acc_B*100:.2f}%")
print(f"C組(5層): {acc_C*100:.2f}%")
print()
print("===== Loss Function 比較總結（皆為4層架構）=====")
print(f"B組(mse): {acc_B*100:.2f}%")
print(f"E組(categorical_crossentropy): {acc_E*100:.2f}%")

===== 層數比較總結 =====
A組(2層): 94.40%
B組(4層): 94.98%
C組(5層): 95.12%

===== Loss Function 比較總結（皆為4層架構）=====
B組(mse): 94.98%
E組(categorical_crossentropy): 97.27%


### 觀察與心得（Loss Function比較）
在相同架構(4層、每層32神經元)下，將loss function從mse換成
categorical_crossentropy，驗證正確率從94.98%提升到97.27%，
差距達2.29%，明顯大於層數比較階段的差距(0.72%)，顯示這個改動
帶來的是真實的效果提升，而非隨機波動。

這印證了理論上的預期：mse是為迴歸問題(預測連續數值)設計的
損失函數，而這是一個多類別分類問題，categorical_crossentropy
才是針對這種任務設計的損失函數，能更準確地衡量「機率分布」
之間的差異。這次的結果也讓我體會到，選擇符合任務性質的loss
function，比單純增加模型複雜度(層數)更能有效提升表現，而且
這個結論在不同次執行中都穩定成立，不像層數比較那樣容易受隨機
性影響。

###用E組進行測試

In [17]:
loss, acc = model.evaluate(x_test, y_test)
print(f"測試資料正確率: {acc*100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9705 - loss: 0.1091
測試資料正確率: 97.05%


### 最終測試結果
最終採用之架構（4層、每層32神經元、categorical_crossentropy、
SGD）在測試資料上的正確率為97.05%，與訓練過程中的驗證正確率
(97.27%)相當接近，顯示模型具備良好的泛化能力，並未出現明顯
過擬合。

###讓模型針對整份測試資料，一次性算出所有預測答案，然後存起來

In [18]:
predict = np.argmax(model.predict(x_test), axis=-1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [19]:
predict

array([7, 2, 1, ..., 4, 5, 6])

###以滑桿方式確認測試圖片樣貌

In [20]:
def test(測試編號):
    plt.imshow(x_test[測試編號].reshape(28,28), cmap='Greys')
    print('神經網路判斷為:', predict[測試編號])

In [21]:
interact_manual(test, 測試編號=(0, 9999));

interactive(children=(IntSlider(value=4999, description='測試編號', max=9999), Button(description='Run Interact', …

In [22]:
def resize_image(inp):
    # 圖在 inp["layers"][0]
    image = np.array(inp["layers"][0], dtype=np.float32)
    image = image.astype(np.uint8)

    # 轉成 PIL 格式
    image_pil = Image.fromarray(image)

    # Alpha 通道設為白色, 再把圖從 RGBA 轉成 RGB
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3]) # 把圖片粘貼到白色背景上，使用透明通道作為遮罩
    image_pil = background

    # 轉換為灰階圖像
    image_gray = image_pil.convert("L")

    # 將灰階圖像縮放到 28x28, 轉回 numpy array
    img_array = np.array(image_gray.resize((28, 28), resample=Image.LANCZOS))

    # 配合 MNIST 數據集
    img_array = 255 - img_array

    # 拉平並縮放
    img_array = img_array.reshape(1, 784) / 255.0

    return img_array

In [23]:
def recognize_digit(inp):
    img_array = resize_image(inp)
    prediction = model.predict(img_array).flatten()
    labels = list('0123456789')
    return {labels[i]: float(prediction[i]) for i in range(10)}

In [ ]:
iface = gr.Interface(
    fn=recognize_digit,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="MNIST 手寫辨識",
    description="請在畫板上繪製數字"
)

iface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://13b87aa4f88f5d8851.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


### Gradio測試心得
實際在畫板上手寫幾個數字進行測試，模型大致都能正確辨識，
但發現筆畫較細、或數字寫在畫布邊緣位置時，辨識信心（機率）
會明顯降低，甚至偶爾誤判。這可能是因為手寫輸入的筆觸粗細、
位置與MNIST訓練資料的風格有落差，即使前面測試資料正確率高達
97%，也不代表對任何手寫風格都能完美辨識，這讓我體會到訓練
資料的分布(distribution)跟實際使用情境是否一致，是模型能否
真正好用的關鍵之一。